# Phenotypic covariance against relatedness (GRM, PC-Relate, Weir-Goudet)
Per phenotype and estimator: least-squares line of y_i y_j on relatedness (2 × kinship) over all pairs (slope, intercept, r²), and pairs binned by kinship. Bins with fewer than 21 pairs are not shown.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
STAGE = "kin_pheno_cov"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
ids = pd.read_csv(target_items(cfg, "grm_shards")[0][0].split(".grm.bin.")[0] + ".grm.id", sep="\t", header=None, dtype=str)[1]
n = len(ids)
est = {"GRM": np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])}   # relatedness: 2 x kinship
for lab, st in [("PC-Relate", "pcrelate"), ("Weir-Goudet", "weir_goudet")]:
    u = single_target(cfg, st)
    if os.path.exists(u):
        assert list(pd.read_csv(u.replace(".bin", ".id"), header=None, dtype=str)[0]) == list(ids), st
        est[lab] = 2 * np.fromfile(u, dtype="float32")
print(n, "people;", ", ".join(est))
i, j = np.tril_indices(n, -1)
off = np.ones(n * (n + 1) // 2, bool); off[np.arange(1, n + 1) * (np.arange(1, n + 1) + 1) // 2 - 1] = False
est = {k: v[off] for k, v in est.items()}; del off
EDGES = 2 * np.unique(np.round(np.r_[np.arange(-0.05, 0.06, 0.005), np.arange(0.06, 0.2, 0.02), [0.2, 0.3, 0.4, 0.6]], 4))
UNREL = 0.1   # "unrelated" fit: pairs with relatedness below this

In [ ]:
sys.path.insert(0, os.path.join(REPO, "src"))
from he_estimators import t_crit
NB, SEED = cfg.param("nblocks"), cfg.param("seed")
block = np.empty(n, int); block[np.random.default_rng(SEED).permutation(n)] = np.arange(n) * NB // n   # people in NB random blocks
bi, bj = block[i], block[j]
NBIN = len(EDGES) - 1
TC = t_crit(NB - 1)

def drop_sums(v, b1, b2, w=None):
    """Per block: sum of v over pairs touching that block (as grm_class_tool's delete-block accumulators)."""
    return np.bincount(b1, v, NB) + np.bincount(b2, np.where(b1 != b2, v, 0), NB)

def fit_moments(M):
    """intercept, slope, r2 from moments [n, Sk, Skk, Sc, Scc, Skc] (last axis)."""
    nn, sk, skk, sc, scc, skc = np.moveaxis(M, -1, 0)
    vk, vc, ck = skk - sk ** 2 / nn, scc - sc ** 2 / nn, skc - sk * sc / nn
    b = ck / vk
    return sc / nn - b * sk / nn, b, ck ** 2 / (vk * vc)

def ranks(v):
    """Ranks 0..n-1 (ties broken by order; pair values are continuous)."""
    r = np.empty(len(v)); r[np.argsort(v, kind="stable")] = np.arange(len(v)); return r

def jk(full, reps):
    """Estimate, delete-block jackknife SE, t interval on NB - 1 df."""
    se = np.sqrt((NB - 1) / NB * ((reps - reps.mean(0)) ** 2).sum(0))
    return full, se, full - TC * se, full + TC * se

rows, binned = [], {}
for uri, item in target_items(cfg, "phenotypes"):
    y = pd.read_csv(uri, sep=" ", dtype={"IID": str}).set_index("IID")["Y"].reindex(ids).to_numpy()
    have = ~np.isnan(y[i]) & ~np.isnan(y[j])
    c = (y[i] * y[j])[have].astype("float64")
    b1, b2 = bi[have], bj[have]
    for lab, kk in est.items():
        k = kk[have].astype("float64")
        for scope, m in [("all pairs", np.ones(len(k), bool)), (f"relatedness < {UNREL}", k < UNREL)]:
            km_, cm_, s1, s2 = k[m], c[m], b1[m], b2[m]
            vals = [np.ones_like(km_), km_, km_ * km_, cm_, cm_ * cm_, km_ * cm_]
            full = np.array([v.sum() for v in vals])
            reps = full - np.stack([drop_sums(v, s1, s2) for v in vals], -1)
            (a, sa, alo, ahi), (b, sb, blo, bhi), (r2, sr, rlo, rhi) = [jk(f, r) for f, r in zip(fit_moments(full), fit_moments(reps))]
            rk, rc_ = ranks(km_), ranks(cm_)   # Spearman: ranks from the full data, jackknifed with the same blocks
            vals = [np.ones_like(rk), rk, rk * rk, rc_, rc_ * rc_, rk * rc_]
            full = np.array([v.sum() for v in vals])
            reps = full - np.stack([drop_sums(v, s1, s2) for v in vals], -1)
            sp, ssp, splo, sphi = jk(fit_moments(full)[2], fit_moments(reps)[2])
            rows.append(dict(**item, estimator=lab, pairs=scope, n_people=int((~np.isnan(y)).sum()), n_pairs=int(m.sum()),
                             intercept=a, intercept_se=sa, intercept_lo=alo, intercept_hi=ahi,
                             slope=b, slope_se=sb, slope_lo=blo, slope_hi=bhi, r2=r2, r2_se=sr, r2_lo=rlo, r2_hi=rhi,
                             spearman_r2=sp, spearman_r2_se=ssp, spearman_r2_lo=splo, spearman_r2_hi=sphi))
        g = np.digitize(k, EDGES) - 1; ok = (g >= 0) & (g < NBIN)
        g, kk2, cc, s1, s2 = g[ok], k[ok], c[ok], b1[ok], b2[ok]
        cnt = np.bincount(g, minlength=NBIN).astype(float)
        sk_, sc_ = np.bincount(g, kk2, NBIN), np.bincount(g, cc, NBIN)
        dn = lambda v: (np.bincount(s1 * NBIN + g, v, NB * NBIN) + np.bincount(s2 * NBIN + g, np.where(s1 != s2, v, 0), NB * NBIN)).reshape(NB, NBIN)
        rc, rn = sc_ - dn(cc), cnt - dn(np.ones_like(cc))
        with np.errstate(invalid="ignore", divide="ignore"):
            _, se, _, _ = jk(sc_ / cnt, rc / rn)
            binned[(item["pheno"], item["transform"], item["covset"], lab)] = (sk_ / np.maximum(cnt, 1), sc_ / np.maximum(cnt, 1), cnt, se)
    print(os.path.basename(uri), "done")
res = pd.DataFrame(rows)
os.makedirs(os.path.dirname(OUT), exist_ok=True)
with record(cfg, STAGE, OUT) as meta:
    res.to_csv(OUT, sep="\t", index=False)
    meta.update(estimators=list(est), unrelated_cut=UNREL, nblocks=NB, ci="delete-block jackknife, people in random blocks, t on nblocks-1 df")
rep.table(res, "estimates", counts=["n_people", "n_pairs"])
pd.set_option("display.width", 200)
display(res[res["pairs"] == "all pairs"].pivot_table(index=["pheno", "transform", "covset"], columns="estimator", values=["slope", "slope_se", "r2", "spearman_r2"]).round(4))

In [ ]:
COV = "base_pcs" if "base_pcs" in cfg.lists["covsets"] else cfg.lists["covsets"][0]
TR = cfg.lists["transforms"][0]
phenos = cfg.lists["phenotypes"]
cols = {"GRM": "#E69F00", "PC-Relate": "#0072B2", "Weir-Goudet": "#009E73"}
LS = {"GRM": "-", "PC-Relate": "--", "Weir-Goudet": ":"}   # overlapping fit lines stay distinguishable
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.3, "font.size": 10})

def fitrow(ph, lab, scope="all pairs"):
    return res[(res["pheno"] == ph) & (res["transform"] == TR) & (res["covset"] == COV) & (res["estimator"] == lab) & (res["pairs"] == scope)].iloc[0]

from aoucov.report import mask_count
NAME = {ph: f"{ph} (n={mask_count(int(res[(res['pheno'] == ph) & (res['transform'] == TR) & (res['covset'] == COV)]['n_people'].iloc[0])):,})" for ph in phenos}
NC = min(4, len(phenos)); NR = -(-len(phenos) // NC)
for name, xmax, ylim in [("bulk", 0.1, (-0.04, 0.06)), ("full", 0.6, None)]:
    fig, axes = plt.subplots(NR, NC, figsize=(4.2 * NC, 3.6 * NR), squeeze=False, sharex=True)
    for ax, ph in zip(axes.flat, phenos):
        for lab in est:
            km, cm, cnt, se = binned[(ph, TR, COV, lab)]
            s = cnt >= 21
            r = fitrow(ph, lab)
            xx = np.array([EDGES[0], xmax])
            ax.plot(xx, r["intercept"] + r["slope"] * xx, color=cols[lab], lw=1.4, ls=LS[lab], zorder=1)   # lines behind points
            ax.errorbar(km[s], cm[s], yerr=TC * se[s] if name == "full" else None, fmt="o", ms=4.5, color=cols[lab], mec="white", mew=0.5, elinewidth=0.8, capsize=0, zorder=2, label=lab)
        ax.axhline(0, color="k", lw=0.6); ax.axvline(0, color="k", lw=0.6)
        ax.set_xlim(EDGES[0], xmax)
        if ylim:
            ax.set_ylim(*ylim)
        ax.set_title(NAME[ph], fontweight="bold"); ax.legend(fontsize=7, frameon=False, loc="upper left")
    for ax in axes.flat[len(phenos):]:
        ax.set_visible(False)
    for ax in axes[-1]:
        ax.set_xlabel("relatedness")
    for ax in axes[:, 0]:
        ax.set_ylabel("mean $y_i y_j$")
    fig.suptitle(f"{RUN_ID} ({TR}, {COV})", fontsize=11)
    fig.tight_layout()
    rep.fig(fig, f"binned_covariance_{name}", f"{TR}/{COV}; relatedness = 2 x kinship; bins with 21+ pairs; lines: least-squares fit over all pairs" + ("; bars: 95% jackknife CI" if name == "full" else "")); plt.show()
bt = pd.concat([pd.DataFrame({"pheno": ph, "estimator": lab, "lo": EDGES[:-1], "hi": EDGES[1:], "mean_rel": km, "mean_cov": cm, "se": se, "pairs": cnt})
                for (ph, tr_, cov_, lab), (km, cm, cnt, se) in binned.items() if tr_ == TR and cov_ == COV])
rep.table(bt[bt["pairs"] >= 21], "binned_covariance", counts=["pairs"], caption=f"{TR}/{COV}: bin means of y_i y_j, bins with 21+ pairs")

In [ ]:
scopes = list(res["pairs"].unique())
y0 = np.arange(len(phenos))[::-1]
fig, axes = plt.subplots(2, 2, figsize=(13, 2 * (1.1 * len(phenos) + 1.2)), sharey=True)
axes = axes.ravel()
for ax, (v, title) in zip(axes, [("slope", "slope"), ("intercept", "intercept"), ("r2", "Pearson r2"), ("spearman_r2", "Spearman r2")]):
    for e, lab in enumerate(est):
        for s, scope in enumerate(scopes):
            d = res[(res["transform"] == TR) & (res["covset"] == COV) & (res["estimator"] == lab) & (res["pairs"] == scope)].set_index("pheno").reindex(phenos)
            yy = y0 + (1 - e) * 0.27 + (0.5 - s) * 0.12
            lo, hi = d[f"{v}_lo"], d[f"{v}_hi"]
            if "r2" in v:
                lo = lo.clip(lower=d[v] * 1e-3)   # log axis: intervals reaching 0 drawn to the edge
            ax.errorbar(d[v], yy, xerr=[d[v] - lo, hi - d[v]], fmt="o" if s == 0 else "s", ms=6.5, color=cols[lab],
                        mfc=cols[lab] if s == 0 else "white", mew=1.5, elinewidth=1, capsize=0, label=f"{lab}: {scope}")
    if "r2" not in v:
        ax.axvline(0, color="k", lw=0.6)
    ax.set_title(title, fontweight="bold")
    ax.set_yticks(y0, [NAME[ph] for ph in phenos]); ax.tick_params(axis="y", labelleft=True)
    ax.grid(axis="y", alpha=0)
for y in y0[::2]:
    for ax in axes:
        ax.axhspan(y - 0.5, y + 0.5, color="0.94", zorder=0)
axes[0].set_yticks(y0, [NAME[ph] for ph in phenos]); axes[0].set_ylim(-0.6, len(phenos) - 0.4)
axes[1].ticklabel_format(axis="x", style="sci", scilimits=(-3, 3))
for ax in axes[2:]:
    ax.set_xscale("log"); ax.xaxis.set_minor_formatter(plt.NullFormatter())
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=len(est), fontsize=9, frameon=False, bbox_to_anchor=(0.5, 0))
fig.suptitle(f"{RUN_ID} ({TR}, {COV})", fontsize=11)
fig.tight_layout(rect=(0, 0.06, 1, 1))
rep.fig(fig, "fit_coefficients", f"least-squares fit of y_i y_j on relatedness (2 x kinship), {TR}/{COV}; filled: all pairs, open: relatedness < {UNREL}; bars: 95% jackknife CI"); plt.show()
rep.save()